# 01 · Environment setup
> **Status:** created locally, syntax-checked only. **NOT yet run on Colab** — update docs/PROGRESS.md after the first real run.

Runtime → Change runtime type → GPU. Upload `train+public_test.zip` to `MyDrive/lifemate/` first (see docs/COLAB.md).

In [ ]:
!nvidia-smi

In [ ]:
# Bootstrap: mount Drive, clone/update repo (token only from Colab Secrets, never hardcoded)
REPO_URL = "https://github.com/Quang1104/lifemate_superklang.git"
BRANCH = "claude/zalo-lifemate-handoff-5c2f7d"
REPO = "/content/lifemate_superklang"
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace("https://", "https://x-access-token:" + userdata.get("GITHUB_TOKEN") + "@")
except Exception:
    pass  # public repo, or secret GITHUB_TOKEN not set
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only", url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
from src.bench import colab_setup as cs, evaluate
cfg = cs.load_config()
print(cs.gpu_report())
DATA_DIR = cs.restore_dataset(cfg)          # unzip Drive archive -> runtime disk (skipped if present)
RESULTS = cs.results_dir(cfg)                # Drive: /MyDrive/lifemate/results
GOLD = evaluate.ensure_gold(DATA_DIR, RESULTS / "gold_validation.jsonl")  # TRAIN held-out hashes, CPU
print(DATA_DIR, GOLD)

In [ ]:
import torch, transformers, peft, json
versions = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__}
print(versions); (RESULTS / "env.json").write_text(json.dumps({**versions, **cs.gpu_report()}, indent=2))

Load check: download pinned models to runtime disk, load planner once, count parameters, one tiny generation.

In [ ]:
PLANNER_KEY, ADAPTER, USE_ASR = "planner", None, True
from src.agent.local_model import LocalPlanner, LocalWhisper, initialize_runtime
from src.agent.runner import Limits
initialize_runtime()
planner = LocalPlanner(cs.fetch_model(cfg, PLANNER_KEY), device="cuda", adapter_dir=ADAPTER)
asr = LocalWhisper(cs.fetch_model(cfg, "asr"), device="cuda") if USE_ASR else None
total = planner.parameter_count + (asr.parameter_count if asr else 0)
print(f"LLM {planner.parameter_count:,}  total {total:,}")
assert planner.parameter_count < 5_000_000_000 and total < 6_000_000_000, "LifeMate parameter limits"
import json; print(json.loads((cs.fetch_model(cfg, PLANNER_KEY) / "PINNED.json").read_text()))
tok = planner.tokenizer
ids = tok.apply_chat_template([{"role": "user", "content": "Trả lời bằng JSON: {\"ok\":true}"}], add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
print(tok.decode(planner.model.generate(ids, max_new_tokens=16, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True))

In [ ]:
!LIFEMATE_DATA={DATA_DIR} python -m pytest tests_audit tests_bench -q -p no:cacheprovider